# 01 — Classical RV Validation

For the top CNN binary candidates with multi-epoch APOGEE data, we perform classical
radial-velocity analysis as an independent validation channel. The pipeline:

1. Measure per-visit RVs via cross-correlation (CCF)
2. Search for orbital periods with a Lomb-Scargle periodogram
3. Fit Keplerian orbits to candidates with significant periodicity

This provides a physics-based check on the CNN's spectral-pattern detections.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.table import Table
from pathlib import Path

# Path setup
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.rv import measure_per_visit_rvs, period_search, fit_orbit, keplerian_rv
from src.data import get_visit_spectra, load_allstar_catalog
from src.utils import plot_rv_curve, save_figure
from config import VALIDATION_CONFIG, VIS_CONFIG

%matplotlib inline
plt.rcParams["figure.dpi"] = VIS_CONFIG["dpi"]

print(f"Project root: {PROJECT_ROOT}")

## 1. Select Top CNN Candidates

In [ ]:
# Load CNN test predictions and the labeled sample
preds = np.load(PROJECT_ROOT / "data" / "cnn_test_preds.npz")
cnn_probs = preds["probabilities"]      # CNN binary probability for each test star
test_indices = preds["test_indices"]     # indices into labeled sample

labeled = Table.read(PROJECT_ROOT / "data" / "labeled_sample.fits")
test_sample = labeled[test_indices]

# Require enough visits for meaningful orbit fitting (NVISITS >= 6)
mask_visits = test_sample["NVISITS"] >= 6
candidates = test_sample[mask_visits]
cnn_probs_filt = cnn_probs[mask_visits]

# Sort by CNN probability descending and take top 100
sort_idx = np.argsort(cnn_probs_filt)[::-1]
top100 = candidates[sort_idx[:100]]
top100_probs = cnn_probs_filt[sort_idx[:100]]

print(f"Test set size:                  {len(test_sample)}")
print(f"With NVISITS >= 6:              {mask_visits.sum()}")
print(f"Selected top candidates:        {len(top100)}")
print(f"CNN probability range:          [{top100_probs.min():.3f}, {top100_probs.max():.3f}]")

## 2. Measure Per-Visit RVs

Use cross-correlation against a template spectrum to independently measure
radial velocities for each APOGEE visit. This bypasses the APOGEE pipeline RVs
and gives us our own RV scatter estimate.

In [ ]:
# Measure per-visit RVs for the first 20 candidates (demo subset)
n_demo = min(20, len(top100))
rv_results = []

for i in range(n_demo):
    star = top100[i]
    apogee_id = star["APOGEE_ID"].strip()
    telescope = star["TELESCOPE"].strip() if "TELESCOPE" in star.colnames else "apo25m"
    field = star["FIELD"].strip() if "FIELD" in star.colnames else None

    try:
        visit = get_visit_spectra(apogee_id, telescope=telescope, field=field)
        # Use the combined spectrum as the CCF template
        template = visit["flux_combined"]
        result = measure_per_visit_rvs(visit, template)

        our_scatter = np.std(result["rv"])
        apogee_vscatter = star["VSCATTER"] if "VSCATTER" in star.colnames else np.nan

        rv_results.append({
            "APOGEE_ID": apogee_id,
            "NVISITS": visit["n_visits"],
            "our_rv_scatter": our_scatter,
            "apogee_vscatter": apogee_vscatter,
            "cnn_prob": top100_probs[i],
            "mjd": result["mjd"],
            "rv": result["rv"],
            "rv_err": result["rv_err"],
        })
        print(f"  [{i+1:2d}/{n_demo}] {apogee_id}: {visit['n_visits']} visits, "
              f"RV scatter = {our_scatter:.2f} km/s")
    except Exception as e:
        print(f"  [{i+1:2d}/{n_demo}] {apogee_id}: FAILED — {e}")

print(f"\nSuccessfully measured RVs for {len(rv_results)}/{n_demo} candidates")

# Summary table
summary = pd.DataFrame([{
    "APOGEE_ID": r["APOGEE_ID"],
    "NVISITS": r["NVISITS"],
    "Our_RV_scatter": f"{r['our_rv_scatter']:.2f}",
    "APOGEE_VSCATTER": f"{r['apogee_vscatter']:.2f}",
    "CNN_prob": f"{r['cnn_prob']:.3f}",
} for r in rv_results])
summary

## 3. Period Search

Run a Lomb-Scargle periodogram on each candidate's RV time series to search
for orbital periods. Flag detections with false-alarm probability < 0.01.

In [ ]:
# Period search for each candidate with measured RVs
FAP_THRESHOLD = 0.01

for r in rv_results:
    ps = period_search(r["mjd"], r["rv"], r["rv_err"])
    r["best_period"] = ps["best_period"]
    r["best_power"] = ps["best_power"]
    r["fap"] = ps["fap"]
    r["frequency"] = ps["frequency"]
    r["power"] = ps["power"]
    r["significant"] = ps["fap"] < FAP_THRESHOLD

n_sig = sum(r["significant"] for r in rv_results)
print(f"Significant period detections (FAP < {FAP_THRESHOLD}): {n_sig}/{len(rv_results)}")

# Separate into strong and weak detections for plotting
strong = [r for r in rv_results if r["significant"]]
weak = [r for r in rv_results if not r["significant"]]

# Plot 4 example periodograms: 2 strong, 2 weak
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
examples = strong[:2] + weak[:2]
titles = ["Strong detection"] * min(2, len(strong)) + ["No significant period"] * min(2, len(weak))

for ax, ex, title in zip(axes.ravel(), examples, titles):
    period_grid = 1.0 / ex["frequency"]
    ax.semilogx(period_grid, ex["power"], "k-", linewidth=0.5)
    ax.axvline(ex["best_period"], color="r", linestyle="--", alpha=0.7,
               label=f"P = {ex['best_period']:.1f} d")
    ax.set_xlabel("Period (days)")
    ax.set_ylabel("LS Power")
    ax.set_title(f"{ex['APOGEE_ID']}\n{title} (FAP = {ex['fap']:.1e})")
    ax.legend(fontsize=8)

plt.tight_layout()
os.makedirs(PROJECT_ROOT / "figures", exist_ok=True)
save_figure(fig, "periodograms")
plt.show()

## 4. Keplerian Orbit Fitting

In [ ]:
# Fit Keplerian orbits for candidates with significant periods
orbit_results = []

for r in rv_results:
    if not r["significant"]:
        r["orbit"] = None
        continue

    try:
        orb = fit_orbit(r["mjd"], r["rv"], r["rv_err"], r["best_period"])
        r["orbit"] = orb
        orbit_results.append(r)
        print(f"{r['APOGEE_ID']}: P={orb['period']:.1f} d, K={orb['K']:.2f} km/s, "
              f"e={orb['ecc']:.3f}, chi2r={orb['reduced_chi2']:.2f}")
    except Exception as e:
        r["orbit"] = None
        print(f"{r['APOGEE_ID']}: fit failed — {e}")

# Print summary table
if orbit_results:
    orbit_df = pd.DataFrame([{
        "APOGEE_ID": r["APOGEE_ID"],
        "Period (d)": f"{r['orbit']['period']:.2f}",
        "K (km/s)": f"{r['orbit']['K']:.2f}",
        "Eccentricity": f"{r['orbit']['ecc']:.3f}",
        "Reduced chi2": f"{r['orbit']['reduced_chi2']:.2f}",
    } for r in orbit_results])
    display(orbit_df)
else:
    print("No successful orbit fits.")

In [ ]:
# Plot the 4 best orbit fits (lowest reduced chi2)
orbit_sorted = sorted(orbit_results, key=lambda r: r["orbit"]["reduced_chi2"])
n_plot = min(4, len(orbit_sorted))

if n_plot > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for idx, ax in enumerate(axes.ravel()):
        if idx >= n_plot:
            ax.set_visible(False)
            continue

        r = orbit_sorted[idx]
        orb = r["orbit"]

        # Generate smooth model curve
        t_model = np.linspace(r["mjd"].min() - 10, r["mjd"].max() + 10, 500)
        rv_model = keplerian_rv(
            t_model, orb["period"], orb["K"], orb["ecc"],
            orb["omega"], orb["T0"], orb["gamma"]
        )

        plot_rv_curve(
            r["mjd"], r["rv"], r["rv_err"],
            title=(f"{r['APOGEE_ID']}\n"
                   f"P={orb['period']:.1f} d, K={orb['K']:.1f} km/s, "
                   f"e={orb['ecc']:.2f}, $\\chi^2_\\nu$={orb['reduced_chi2']:.2f}"),
            model_mjd=t_model,
            model_rv=rv_model,
            ax=ax,
        )

    plt.tight_layout()
    save_figure(fig, "orbit_fits")
    plt.show()
else:
    print("No orbit fits to plot.")

## 5. Validation Summary

In [ ]:
# Classify each candidate based on RV analysis
RV_SCATTER_THRESHOLD = 1.0  # km/s — minimum scatter to consider variable
CHI2_THRESHOLD = 3.0        # reduced chi2 threshold for a good orbit fit

for r in rv_results:
    has_period = r["significant"]
    good_fit = (r["orbit"] is not None and r["orbit"]["reduced_chi2"] < CHI2_THRESHOLD)
    high_scatter = r["our_rv_scatter"] > RV_SCATTER_THRESHOLD

    if has_period and good_fit:
        r["classification"] = "confirmed"
    elif has_period and not good_fit:
        r["classification"] = "likely"
    elif not has_period and high_scatter:
        r["classification"] = "ambiguous"
    else:
        r["classification"] = "non-variable"

# Count and display
from collections import Counter
counts = Counter(r["classification"] for r in rv_results)
total = len(rv_results)

print("RV Validation Classification")
print("=" * 45)
for cls in ["confirmed", "likely", "ambiguous", "non-variable"]:
    n = counts.get(cls, 0)
    print(f"  {cls:15s}: {n:3d}  ({100*n/total:.1f}%)")
print("-" * 45)
print(f"  {'total':15s}: {total:3d}")
print()

confirmed_frac = counts.get("confirmed", 0) / total if total > 0 else 0
print(f"Key result: {100*confirmed_frac:.0f}% of CNN candidates confirmed by independent RV analysis")
print(f"            {100*(counts.get('confirmed',0)+counts.get('likely',0))/total:.0f}% "
      f"confirmed or likely (significant period detected)")

In [ ]:
# Save validation results to CSV
output_rows = []
for r in rv_results:
    orb = r["orbit"]
    output_rows.append({
        "APOGEE_ID": r["APOGEE_ID"],
        "cnn_prob": r["cnn_prob"],
        "vscatter": r["our_rv_scatter"],
        "best_period": r["best_period"],
        "fap": r["fap"],
        "K": orb["K"] if orb else np.nan,
        "ecc": orb["ecc"] if orb else np.nan,
        "reduced_chi2": orb["reduced_chi2"] if orb else np.nan,
        "classification": r["classification"],
    })

val_df = pd.DataFrame(output_rows)
outpath = PROJECT_ROOT / "data" / "rv_validation_results.csv"
val_df.to_csv(outpath, index=False)
print(f"Saved {len(val_df)} rows to {outpath}")
val_df.head(10)